In [ ]:
import numpy as np
import tensorflow as tf
from collections import deque

model = tf.keras.models.load_model("helmX_autoencoder.keras")

mean = np.load("processed/mean.npy")
std  = np.load("processed/std.npy")
threshold = np.load("processed/threshold.npy")

WINDOW_SIZE = 60
STRIDE = 15

buffer = deque(maxlen=WINDOW_SIZE)
new_samples = 0

from mpu6050 import mpu6050

sensor = mpu6050(0x68)

def read_sensor():
    accel = sensor.get_accel_data()
    gyro  = sensor.get_gyro_data()

    return [
        accel['x'], accel['y'], accel['z'],
        gyro['x'],  gyro['y'],  gyro['z']
    ]


import time

while True:
    sample = read_sensor()
    buffer.append(sample)
    new_samples += 1

    if len(buffer) < WINDOW_SIZE:
        time.sleep(0.01)
        continue

    if new_samples < STRIDE:
        time.sleep(0.01)
        continue

    new_samples = 0

    # ---- prediction block ----
    window = np.array(buffer)
    window = (window - mean) / std
    window = np.transpose(window, (1,0))
    window = window[np.newaxis, ..., np.newaxis]

    recon = model.predict(window, verbose=0)
    mse = np.mean((window - recon)**2)

    if mse > threshold:
        print("🚨 Crash detected")

    time.sleep(0.01)
